
# Stage E — “chats & ACLs”

**Goal:** minimal but correct separation.

* **Pre-booking chat**: create/find `PreBookingConversation(tourId, motardId, participantId)`; `POST /prebooking/messages`.
* **Booking chat**: `BookingConversation(bookingId)`; `POST /booking/messages`.
* Add simple **ACL checks** (only participant/motard can read/write).

**Discuss / decide**

* UI experience: show a single thread with a divider when booking is created (backend still stores two conversations).
* Rate limit + spam guard.

**DoD**

* You can post/read messages for both contexts; wrong users get 403.



# Stage F — "auth, roles, and guardrails"

**Goal:** make it realistic for class.

* Identity setup with seed users.
* Policy attributes: `[Authorize(Roles="Motard")]` for tour creation & approval.
* `[Authorize]` for booking/chat.

**Decisions Made:**

* ✅ **Auth Mode**: Cookie-based authentication (simpler for class demo)
* ✅ **Testing**: No UI needed - use Swagger/Postman with seeded users
* ✅ **Role Model**:
  - **Organizer** (1 user): Has BOTH "Organizer" + "Motard" roles - can create tours AND ride
  - **Pure Motards** (2-3 users): Only "Motard" role - can request to join tours (needs organizer approval)
  - **Participants** (2-3 users): Only "Participant" role - can only book tours
* ✅ **Motard Join Flow**: Option B - Requires organizer approval to join a tour
* 🔄 **Motard Profile Requirements**: TBD - require `licenseNo`?

**Implementation Steps:**

1. ✅ **Configure cookie authentication** - DONE
   - Identity configured with relaxed password rules for demo
   - Cookie settings: 24h expiry, returns 401/403 instead of redirects
   - Authentication middleware enabled
   - Auth controller created with `/api/auth/login`, `/api/auth/logout`, `/api/auth/me`
   
2. ✅ **Seed users with roles** - DONE
   - 1 Organizer: `organizer1` (roles: Organizer + Motard)
   - 2 Motards: `motard1`, `motard2` (role: Motard only)
   - 2 Participants: `participant1`, `participant2` (role: Participant)
   - All passwords: `pass`
   - Demo tour seeded with organizer + motard1 as riders
   
3. 🔄 **Add [Authorize] attributes to controllers** - IN PROGRESS
   - ✅ **Step 3A**: Tour creation endpoint added (`POST /api/tours`)
     - Only users with "Motard" role can create
     - Creator becomes first motard/organizer
     - Requires motard profile to be complete
   - ⬜ **Step 3B**: Protect booking approval (only assigned motard)
   - ⬜ **Step 3C**: Protect chat endpoints (only conversation participants)

4. ⬜ Test authorization enforcement

**Seeded Users (for testing):**
| Username | Password | Roles | Can Do |
|----------|----------|-------|--------|
| `organizer1` | `pass` | Organizer, Motard | Create tours, ride as motard |
| `motard1` | `pass` | Motard | Create tours, request to join, accept bookings |
| `motard2` | `pass` | Motard | Create tours, request to join, accept bookings |
| `participant1` | `pass` | Participant | Browse tours, book with motards |
| `participant2` | `pass` | Participant | Browse tours, book with motards |

**DoD**

* Unauthorized calls blocked; role-guarded endpoints enforced.

---

Excellent! 🎉 **Step 3C chat authorization tests all passed!**

**Confirmed working:**
- ✅ Anonymous users → 401 Unauthorized
- ✅ Participants can only access their own conversations → 403 Forbidden for others
- ✅ Motards can only access conversations they're part of → 403 Forbidden for others
- ✅ Correct participants can send/read messages → 200/201 OK
- ✅ Each participant can create their own separate conversation with a motard

---

## **Stage F — Authentication & Authorization: COMPLETE! ✅**

**Summary of what was implemented:**

### **Step 1: Cookie Authentication Setup**
- ✅ ASP.NET Core Identity with cookie-based authentication
- ✅ 24-hour sliding expiration
- ✅ API-friendly 401/403 responses (no redirects)

### **Step 2: User Seeding with Roles**
- ✅ 3 roles: Organizer, Motard, Participant
- ✅ 5 users: organizer1 (both roles), motard1/motard2, participant1/participant2
- ✅ All passwords: "pass"

### **Step 3A: Tour Creation Authorization**
- ✅ `[Authorize(Roles="Motard")]` on POST /api/tours
- ✅ Only motards can create tours
- ✅ Validates motard profile exists

### **Step 3B: Booking Approval Authorization**
- ✅ `[Authorize(Roles="Motard")]` on booking approval
- ✅ Only the **assigned motard** can approve their bookings
- ✅ Booking requests use authenticated user (no hardcoded participants)

### **Step 3C: Chat Endpoint Authorization**
- ✅ `[Authorize]` on entire ChatsController
- ✅ Only conversation participants (motard + participant) can access messages
- ✅ Separate conversations are properly isolated
- ✅ Added helper endpoints to list user's conversations

---

**All authorization requirements met:**
- ✅ Identity setup with seed users
- ✅ Policy attributes enforced
- ✅ Unauthorized calls blocked (401)
- ✅ Role-guarded endpoints enforced (403)
- ✅ Conversation access restricted to participants

**Would you like to:**
1. Document this in the notebook?
2. Run the automated unit tests?
3. Move on to the next stage?

## 🧪 How to Test with REST Client Extension

### Installation
**VS Code Extension:** Search for "REST Client" by Huachao Mao in Extensions (Ctrl+Shift+X)

### Test Files Available
- `test_auth.http` - Authentication tests (login, logout, check current user)
- `test_tour_creation.http` - Tour creation with role-based authorization tests

### How to Use
1. **Start the app:** Open terminal → `dotnet run` (in `moto_micro_tours` folder)
2. **Open test file:** Click on `test_auth.http` or `test_tour_creation.http`
3. **Send requests:** Click **"Send Request"** link that appears above each request
4. **View results:** Response appears in a split panel on the right
5. **Cookies automatic:** Login once, all subsequent requests use that session

### Quick Test Flow
```http
1. POST /api/auth/login (as motard1)      → 200 OK
2. POST /api/tours (create a tour)        → 201 Created
3. GET /api/tours (verify it appears)     → 200 OK with tour list
4. POST /api/auth/logout                  → 200 OK
```

### Expected Authorization Results
| User Type | Can Login? | Can Create Tours? | Expected Response |
|-----------|------------|-------------------|-------------------|
| Anonymous | N/A | ❌ No | 401 Unauthorized |
| Participant | ✅ Yes | ❌ No | 403 Forbidden |
| Motard | ✅ Yes | ✅ Yes | 201 Created |
| Organizer | ✅ Yes | ✅ Yes (has Motard role) | 201 Created |

### Tips
- **No copy/paste needed** - REST Client handles cookies automatically
- **Run in order** - Especially in `test_tour_creation.http`
- **Multiple users** - Logout between tests to switch users
- **Validation** - Try invalid data (past dates, negative capacity) to see error handling

---


# Stage G — “cancellation/refund & reviews (light)”

**Goal:** round out the flow.

* `POST /bookings/{id}/cancel` (participant before payment → free; after payment → non-refundable or partial, per rule)
* `POST /bookings/{id}/refund` (organizer cancel → full refund (mock), payout status adjusts)
* `POST /motards/{id}/reviews` (if participant had a Paid booking in the past for this motard)

**Discuss / decide**

* A simple rule table you can show in class (who cancels? when? outcome?).
* Review moderation (skip in MVP).

**DoD**

* Simple state transitions work; payouts flip to `Pending/Failed` if refunded (mock logic).

---

# Stage H — “demo storyboard & test checklist”

**Goal:** teacher-friendly demo that hits the learning objectives.

**Storyboard (5–7 minutes)**

1. Show 1 tour with **2 motards** (capacity 1 vs 3; price 20€ vs 10€).
2. As participant: open motard profile → pre-booking chat question.
3. Send **booking request** to motard A (1-to-1).
4. As motard A: **approve** → system returns **payment link** (mock).
5. As participant: **pay** → booking turns **Paid**, **Payout 50/50** created.
6. Switch to **booking chat** to agree meeting point.
7. (Optional) Organizer cancels another booking → mock refund → payout updates.

**Test checklist**

* Capacity: approving beyond capacity returns 400.
* Booking to a **published** tour only.
* Only chosen motard can approve that booking.
* Payment creates payout = exact half/half of seat price.
* Chat visibility restricted by role & context.
* Cancellation rules applied correctly.

---

## Questions for us to iterate (short list)

1. **Approval expiry**: after approval, how long does the participant have to pay? (e.g., 24h)
2. **Cancellation matrix** (who/when/outcome): do you want a 3×3 table now?
3. **Minimum motard profile**: require `licenseNo` before listing?
4. **Rate limits**: max pre-booking messages/conversations per hour?
5. **Booking identifier** formatting for the UI (e.g., `BK-2025-00012`)?
6. **i18n**: EN default with FR/中文 later, or start bilingual?

---

## What to do next (your immediate tasks)

* Put **Stage A Spec v1** (rules + PlantUML) into your ipynb.
* Decide the **cancellation matrix** + **approval expiry** (just numbers!).
* Confirm **auth mode** (cookies vs JWT) for class constraints.

when you have those tiny decisions, i’ll tighten the ruleset (pre/post-conditions per UC + a small state machine for `Booking`/`Payout`) so Copilot can scaffold cleanly.